# <font color="ffc800"> **Tamil Piper Training V6 (The C++ Fix)**

**Changes in V6:**
1.  **Missing Ingredients:** `piper-phonemize` failed because it needs the C++ `onnxruntime` library (not just the Python one). V6 downloads this library manually.
2.  **Hardcoded Linking:** It forces the installer to use these downloaded C++ libraries.

**Why this works:** We are manually providing the "bricks" (C++ libs) that Python 3.12 couldn't find to build the "bridge" (piper-phonemize).

---

### **Instructions:**
1.  Upload `TextyMcSpeechy.zip` to Drive.
2.  Run all cells.

In [ ]:
#@markdown # <font color="ffc800"> **1. Mount Drive & Setup Workspace** 📂
import os
import shutil
import zipfile
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

# --- CONFIGURATION ---
DRIVE_ROOT = "/content/drive/MyDrive"
TEXTY_ZIP = os.path.join(DRIVE_ROOT, "TextyMcSpeechy.zip")
TEXTY_DIR = os.path.join(DRIVE_ROOT, "TextyMcSpeechy")
SOURCE_DATASET = os.path.join(TEXTY_DIR, "tts_dojo/DATASETS/tamil_dataset/wav_22050")
if not os.path.exists(SOURCE_DATASET):
    # Fallback to wavs
    SOURCE_DATASET = os.path.join(TEXTY_DIR, "tts_dojo/DATASETS/tamil_dataset/wavs")

# Local Paths
WORKSPACE_DIR = "/content/workspace"
DATASET_DIR = os.path.join(WORKSPACE_DIR, "dataset")
TRAIN_OUTPUT_DIR = os.path.join(WORKSPACE_DIR, "piper_train_tamil")

# Pre-cleanup
if os.path.exists(WORKSPACE_DIR):
    shutil.rmtree(WORKSPACE_DIR)
os.makedirs(DATASET_DIR)
os.makedirs(os.path.join(DATASET_DIR, "wavs"))

# --- PREPARE DATA ---
# 1. Unzip if needed
if not os.path.exists(TEXTY_DIR):
    if os.path.exists(TEXTY_ZIP):
        print(f"Unzipping {TEXTY_ZIP}...")
        with zipfile.ZipFile(TEXTY_ZIP, 'r') as zip_ref:
            zip_ref.extractall(DRIVE_ROOT)
    else:
        print("⚠️ WARNING: TextyMcSpeechy not found in Drive!")

# 2. Copy Metadata
meta_src = os.path.join(TEXTY_DIR, "tts_dojo/DATASETS/tamil_dataset/metadata.csv")
if os.path.exists(meta_src):
    shutil.copy(meta_src, os.path.join(DATASET_DIR, "metadata.csv"))
    print("✅ Metadata copied.")
else:
    raise FileNotFoundError("❌ ERROR: metadata.csv missing!")

# 3. Link Audio
if os.path.exists(SOURCE_DATASET):
    print("Copying audio files...")
    !cp -r "{SOURCE_DATASET}"/* "{DATASET_DIR}/wavs/"
    print(f"✅ Audio files prepared.")
else:
    raise FileNotFoundError("❌ ERROR: Audio folder missing!")

In [ ]:
#@markdown # <font color="ffc800"> **2. Install Dependencies (C++ patched)** 📦
import os
import sys
import shutil

# 1. Install System Libs
!sudo apt-get update > /dev/null
!sudo apt-get install -y espeak-ng portaudio19-dev libespeak-ng-dev > /dev/null

%cd /content

# 2. Download ONNXRuntime C++ (Missing Ingredient)
ORT_VERSION = "1.14.1"
ORT_FILE = f"onnxruntime-linux-x64-{ORT_VERSION}.tgz"
ORT_URL = f"https://github.com/microsoft/onnxruntime/releases/download/v{ORT_VERSION}/{ORT_FILE}"

if not os.path.exists(f"onnxruntime-linux-x64-{ORT_VERSION}"):
    print("Downloading onnxruntime C++ library...")
    !wget -q {ORT_URL}
    !tar -xzf {ORT_FILE}
else:
    print("Using cached onnxruntime header files.")

ORT_PATH = os.path.abspath(f"onnxruntime-linux-x64-{ORT_VERSION}")
print(f"OnnxRuntime Path: {ORT_PATH}")

# 3. Clone & Patch Piper Phonemize
if os.path.exists("piper-phonemize"):
    shutil.rmtree("piper-phonemize")

!git clone https://github.com/rhasspy/piper-phonemize.git
%cd piper-phonemize

# --- PATCH SETUP.PY ---
print("Patching setup.py to link missing C++ libs...")
setup_content = """\
from setuptools import setup, Extension
from pathlib import Path

setup(
    name="piper_phonemize",
    version="1.2.0",
    description="Phonemization for Piper TTS",
    author="Michael Hansen",
    author_email="mike@rhasspy.org",
    url="https://github.com/rhasspy/piper-phonemize",
    packages=["piper_phonemize"],
    package_dir={"piper_phonemize": "piper_phonemize"},
    package_data={"piper_phonemize": ["py.typed"]},
    ext_modules=[
        Extension(
            "piper_phonemize_cpp",
            [
                "src/phonemize.cpp",
                "src/python.cpp",
            ],
            include_dirs=[f\"{ORT_PATH}/include\", "/usr/include"],
            library_dirs=[f\"{ORT_PATH}/lib\", "/usr/lib"],
            libraries=["espeak-ng", "onnxruntime"],
            extra_compile_args=["-std=c++17"],
            runtime_library_dirs=[f\"{ORT_PATH}/lib\"]
        )
    ],
    classifiers=[
        "Development Status :: 3 - Alpha",
        "Intended Audience :: Developers",
        "Topic :: Text Processing :: Linguistic",
        "License :: OSI Approved :: MIT License",
        "Programming Language :: Python :: 3.7",
        "Programming Language :: Python :: 3.8",
        "Programming Language :: Python :: 3.9",
        "Programming Language :: Python :: 3.10",
    ],
    keywords="phonemize piper tts",
)
""".replace("{ORT_PATH}", ORT_PATH)

with open("setup.py", "w") as f:
    f.write(setup_content)

# Install
print("Compiling piper-phonemize...")
!pip install .

# 4. Install Piper Train
%cd /content
if os.path.exists("piper"):
    shutil.rmtree("piper")
!git clone https://github.com/rhasspy/piper.git
%cd piper/src/python

!pip install "numpy<2.0"
!sed -i 's/piper-phonemize~=1.1.0/piper-phonemize>=1.1.0/' requirements.txt
!sed -i 's/pytorch-lightning~=1.7.0//' requirements.txt
!pip install "pytorch-lightning==1.9.5" "torchmetrics>=0.7.0"
!pip install onnx onnxruntime
!pip install -e .
!./build_monotonic_align.sh

print("✅ Environment Installed Successfully!")

In [ ]:
#@markdown # <font color="ffc800"> **3. Preprocess Dataset** 🔄
import os

%cd /content/piper/src/python

WORKSPACE_DIR = "/content/workspace"
DATASET_DIR = os.path.join(WORKSPACE_DIR, "dataset")
OUTPUT_DIR = os.path.join(WORKSPACE_DIR, "piper_train_tamil")

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

print("Starting Preprocessing...")
!python -m piper_train.preprocess \
  --language ta \
  --input-dir "{DATASET_DIR}" \
  --output-dir "{OUTPUT_DIR}" \
  --dataset-name "tamil_piper" \
  --dataset-format ljspeech \
  --sample-rate 22050

# Validation
config_path = os.path.join(OUTPUT_DIR, "config.json")
if os.path.exists(config_path):
    print("✅ Preprocessing Successful!")
else:
    raise FileNotFoundError(f"❌ Preprocessing Failed! Check output above.")

In [ ]:
#@markdown # <font color="ffc800"> **4. Train & Auto-Backup** 🏋️‍♂️
import os

WORKSPACE_DIR = "/content/workspace"
OUTPUT_DIR = os.path.join(WORKSPACE_DIR, "piper_train_tamil")
DRIVE_BACKUP = "/content/drive/MyDrive/piper_train_tamil/checkpoints"
if not os.path.exists(DRIVE_BACKUP):
    os.makedirs(DRIVE_BACKUP)

%load_ext tensorboard
%tensorboard --logdir "{OUTPUT_DIR}/lightning_logs"

print("Starting Training...")

!python -m piper_train \
    --dataset-dir "{OUTPUT_DIR}" \
    --accelerator gpu \
    --devices 1 \
    --batch-size 8 \
    --validation-split 0.0 \
    --num-test-examples 0 \
    --max_epochs 100 \
    --checkpoint-epochs 5 \
    --precision 32 \
    --quality medium \
    --callbacks.default_checkpoint_monitor.dirpath "{DRIVE_BACKUP}"

!cp -r "{OUTPUT_DIR}/lightning_logs" "/content/drive/MyDrive/piper_train_tamil/"